<a href="https://colab.research.google.com/github/chiraswykj/GENAI_TUTORIALS/blob/main/GENAI77.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -qU langchain langchain-community langchain-huggingface langchain-tavily langgraph transformers accelerate gradio
!pip install -qU "requests==2.32.5"

import os

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True,
    return_full_text=False
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    Calculate basic mathematical expressions.
    """
    import ast
    import operator

    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv
    }

    def calculate(node):
        if isinstance(node, ast.Constant):
            if type(node.value) not in (int, float):
                raise ValueError("Invalid number")
            return node.value

        if isinstance(node, ast.BinOp):
            op = operators.get(type(node.op))
            if op is None:
                raise ValueError("Unsupported operator")
            return op(
                calculate(node.left),
                calculate(node.right)
            )
        raise ValueError("Invalid expression")

    return str(
        calculate(
            ast.parse(expression, mode="eval").body
        )
    )

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the following question clearly.

Question: {question}

Answer:
""")

chain = prompt | llm

def ask_agent(question):
    result = chain.invoke({
        "question": question
    })
    return result.strip()

print(ask_agent("Explain artificial intelligence"))

import gradio as gr

def chat(message, history):
    answer = ask_agent(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="Open Source AI Agent",
    description="Powered by LangChain and Qwen",
    examples=[
        "What is AI?",
        "Explain machine learning",
        "What is LangChain?",
        "Explain neural networks"
    ]
)

demo.launch(share=True)
print("Open-source LLM loaded!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It involves creating computer systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI can be categorized into three main types:

1. Narrow or Weak AI: Also known as "conversational" or "narrow" AI, it is designed for specific tasks within a narrow domain. Examples include virtual assistants like Siri and Alexa, which perform tasks based on predefined rules and data.

2. General or Strong AI: This type of AI aims to achieve human-level intelligence across all domains, including understanding natural language, learning from experience, and making decisions without being explicitly programmed. While still largely theoretical, some experts believe this form of AI may eventually become possible.

3. Superintelligence: This hypothetical sce

Open-source LLM loaded!
